# 🎓 Explora - Study Abroad Guide
### CS529 AI Engineering — Day 2 Assignment: AI Chatbot Development Using LLMs and Gradio


In [14]:
from dotenv import load_dotenv
import os
from openai import OpenAI

from IPython.display import Markdown, display

In [15]:
load_dotenv()

True

In [16]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")
import gradio as gr
print(f"Gradio {gr.__version__} loaded.")

Gradio 6.29.0 loaded.


In [17]:
### API key config

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"):
    print(f"API key found (starts with {OPENAI_API_KEY[:9]}...).")
    client = OpenAI(api_key=OPENAI_API_KEY, timeout=20.0)
else:
    print("⚠️ OPENAI_API_KEY not found in .env — see README for how to create one, then re-run this cell.")
    client = None

API key found (starts with sk-proj-g...).


## Prompt engineering
- `prompts/chatbot.txt` — the instruction **template**, with the `{{projectInfo}}` placeholder;
- `prompts/explora.md` — the **project rules** (scope, audience, tone, safety).

- **Role** — a standalone study-abroad guide for the Explora educational project.
- **Goal** — give general, responsible information to international students (largely Angolan) starting to research studying in the United States.
- **Context** — educational project by an Angolan student in the U.S.; young, mobile-first, international audience.
- **Constraints** — strictly U.S. study topics (brief refusal + redirect otherwise); no individualized advice; no legal/immigration/educational/financial advice; no step-by-step application instructions; general information only.
- **Data** — the project rules injected via the `{{projectInfo}}` placeholder.
- **Output** — warm and welcoming tone; answer directly in ≤ 2 short paragraphs.
- **Success criteria** — a response is successful when it answers directly, stays in scope, stays concise, uses only general information without invented details, and keeps a warm tone.

Techniques used:
- **Scope guardrails** — strictly U.S. study topics; brief refusal + redirect for anything else (other countries, living/working/immigrating abroad). Includes a one-shot example showing the refusal tone and format.
- **Audience awareness / fairness** — international, largely Angolan audience; never assume the user's country; neutral language ("school documents"), no country-specific exams (ENEM/SAT/ACT), no assumed degree paths.
- **Anti-hallucination / accuracy** — "never invent information"; if unsure about a fact, say so plainly instead of guessing; redirect personal cases to general explanations.
- **Identity protection** — always an educational project/community, never a company or job opportunity.
- **Privacy** — no credentials or personal data in the prompt; the API key is loaded from `.env`, never hardcoded.
- **Zero-shot + one example** — the prompt is zero-shot throughout except for the single refusal example, per Lesson 3's guidance to use examples only where they improve consistency.

In [18]:
with open("prompts/chatbot.txt", encoding="utf-8") as f:
    template = f.read()
with open("prompts/explora.md", encoding="utf-8") as f:
    project_info = f.read()

system_prompt = (
    template
    .replace("{{projectInfo}}", project_info)
)

assert "{{" not in system_prompt, "Unsubstituted placeholder left in the system prompt!"
print(f"System prompt built: {len(system_prompt)} characters, no placeholders left.")
print("---- preview ----")
print(system_prompt[:900] + "...")

System prompt built: 6641 characters, no placeholders left.
---- preview ----
You are Explora — Study Abroad Guide, a standalone educational chatbot.

- You represent Explora, an educational project created by an Angolan student in the United States.
- Your tone is warm, empathetic, encouraging, and informative.
- You never provide individualized advice, consulting, or step-by-step instructions.
- You always reinforce that your role is to provide general and educational information.
- Your scope is strictly limited to general information about studying in the United States.
- Default to concise answers.
- In most cases, answer in 2 short paragraphs or less.
- Start by answering the user's question directly instead of adding a long introduction.
- Do not repeat the user's question back to them unless it is necessary for clarity.
- Do not repeat the same idea in different words.
- Do not add generic closing lines like "feel free to ask" unless they genuinely help.
-...


In [19]:
### Input validation
MAX_MESSAGE_LENGTH = 1000

def validate_input(message):
    """Validate a user message. Returns (ok, text_or_error)."""
    if message is None:
        return False, "⚠️ Please type a message first."
    text = str(message).strip()
    if not text:
        return False, "⚠️ Your message is empty — try e.g. 'What is campus life like at US universities?'"
    if len(text) > MAX_MESSAGE_LENGTH:
        return False, f"⚠️ Message too long ({len(text)} chars). Please keep it under {MAX_MESSAGE_LENGTH} characters."
    return True, text


# Self-test — no API key needed
cases = [
    ("", False),
    ("   ", False),
    (None, False),
    ("What scholarships exist for international students?", True),
    ("  hello  ", True),
    ("x" * 1001, False),
    ("x" * 1000, True),
]
for raw, expected in cases:
    ok, out = validate_input(raw)
    label = "PASS" if ok == expected else "FAIL"
    preview = repr(str(raw)[:30])
    print(f"{label}: input={preview} -> ok={ok} | {out[:75]}")
    assert ok == expected, f"Validation failed for {preview!r}"
print("\nAll validation tests passed ✅")

PASS: input='' -> ok=False | ⚠️ Your message is empty — try e.g. 'What is campus life like at US univers
PASS: input='   ' -> ok=False | ⚠️ Your message is empty — try e.g. 'What is campus life like at US univers
PASS: input='None' -> ok=False | ⚠️ Please type a message first.
PASS: input='What scholarships exist for in' -> ok=True | What scholarships exist for international students?
PASS: input='  hello  ' -> ok=True | hello
PASS: input='xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx' -> ok=False | ⚠️ Message too long (1001 chars). Please keep it under 1000 characters.
PASS: input='xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx' -> ok=True | xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

All validation tests passed ✅


## Model parameters
- **gpt-4o-mini** — fast and cheap, plenty for short guardrailed answers;
- **temperature 0.2** — low randomness keeps answers factual and consistent with the rules;
- **max 200 tokens** — enforces the "2 short paragraphs" style and caps cost per reply.

In [20]:
## Chat logic

MODEL = "gpt-4o-mini"

def chat(message, history):
    """history: list of {"role": "user"|"assistant", "content": str} (Gradio messages format)."""
    ok, result = validate_input(message)
    if not ok:
        return result  # friendly validation message, no API call spent
    if client is None:
        return "⚠️ The OpenAI API key isn't configured. Add OPENAI_API_KEY to your .env file and re-run the setup cell."
    messages = [{"role": "system", "content": system_prompt}]
    messages += [{"role": h["role"], "content": h["content"]} for h in (history or [])]
    messages.append({"role": "user", "content": result})
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.2,
            max_tokens=200,
        )
        return (response.choices[0].message.content or "").strip()
    except Exception as e:
        msg = str(e).lower()
        if "authentication" in type(e).__name__.lower() or "invalid_api_key" in msg or "unauthorized" in msg:
            return "⚠️ The API key was rejected. Check that OPENAI_API_KEY in your .env file is valid and has credits."
        return "⚠️ Sorry, the assistant service didn't respond. Please try again in a moment."

In [21]:
## Smoke test

if client is None:
    print("Skipping live smoke test — no API key configured. It will run once you add your key and re-run.")
else:
    q = "What is campus life like at universities in the United States?"
    print("You:", q)
    print("Hela:", chat(q, []))

You: What is campus life like at universities in the United States?


Hela: Campus life at universities in the United States is often vibrant and diverse, offering a wide range of activities and experiences. Students typically engage in various clubs, organizations, and events that cater to different interests, from academic and cultural groups to sports and recreational activities. Many campuses also host social events, concerts, and festivals, creating a lively atmosphere where students can connect and build friendships.

Additionally, universities often provide resources for student support, including counseling services, academic advising, and wellness programs. Living arrangements, such as dormitories or student apartments, also play a significant role in campus life, fostering a sense of community. Overall, the experience can vary greatly depending on the institution, but many students find it enriching and fulfilling.


In [ ]:
## Gradio UI

import inspect
_chatbot_kwargs = {"label": "Chat", "height": 420}
# Gradio 5.x needs type="messages" for dict-style history; Gradio 6+ dropped the parameter.
if "type" in inspect.signature(gr.Chatbot.__init__).parameters:
    _chatbot_kwargs["type"] = "messages"
_blocks_kwargs = {"title": "Explora — Study Abroad Guide"}
_launch_kwargs = {"share": True}
# Gradio 6 moved `theme` from Blocks() to launch().
if int(gr.__version__.split(".")[0]) >= 6:
    _launch_kwargs["theme"] = gr.themes.Soft()
else:
    _blocks_kwargs["theme"] = gr.themes.Soft()
with gr.Blocks(**_blocks_kwargs) as demo:
    gr.Markdown("# 🎓 Explora — Study Abroad Guide\n*General, welcoming information for international students researching study in the United States.*")
    chatbot = gr.Chatbot(**_chatbot_kwargs)
    with gr.Row():
        msg = gr.Textbox(label="Your message", placeholder="e.g., What is campus life like at US universities?", scale=4, max_lines=4)
        send_btn = gr.Button("Send ➤", variant="primary", scale=1)
    clear_btn = gr.Button("🧹 Clear chat")
    gr.Examples(
        examples=[
            "What is campus life like at universities in the United States?",
            "Are there scholarships for international students in the US?",
            "What is the academic environment like at US colleges?",
            "What kind of support do US universities offer international students?",
        ],
        inputs=msg,
        label="Try an example question",
    )
    with gr.Accordion("What can I ask about?", open=False):
        gr.Markdown(
            "- 🎓 Universities & colleges in the **United States**\n"
            "- 🏫 Campus life, academic environment, student experience\n"
            "- 💰 Tuition & scholarships (general information)\n"
            "- 🤝 International student support services\n\n"
            "Out of scope: other countries, visas/immigration advice, and step-by-step application instructions."
        )
    gr.Markdown("*CS529 Day 2 assignment · Python/Gradio port of [helenapedro/ai-chatbot-app](https://github.com/compro-miu-helena/study-abroad-guide.git) · Powered by OpenAI gpt-4o-mini*")

    def respond(message, history):
        reply = chat(message, history or [])
        history = (history or []) + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply},
        ]
        return "", history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    send_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(lambda: ([], ""), None, [chatbot, msg])

print("UI built ✅ — run the next cell to launch it and get the shareable link.")

UI built ✅ — run the next cell to launch it and get the shareable link.


In [23]:
demo.launch(**_launch_kwargs)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/09/29 16:13:27 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout
